**Basic GP fit**

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel

df = pd.read_csv("beard_model_ready_features.csv")

feature_cols = [
    "HeavyAtomCount", "LargestConjugatedSystemSize", "TPSA",
    "NumHDonors", "NumRotatableBonds", "FractionCSP3", "MolLogP_clipped",
    "NumDonorGroups", "NumAcceptorGroups", "HasPushPull"
]

X = df[feature_cols]
y = df["lambda_max_exp_nm"]

# Same split as Phase 3 -- same random_state, so the comparison is apples-to-apples
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

Train: (5502, 10), Test: (1376, 10)


**the kernel and fit the first GP**

In [5]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel

# Kernel components, each with a purpose:
# ConstantKernel -- scales the overall variance of the function (how large can predictions swing)
# RBF -- the core similarity function; length_scale controls how "far" in feature space
#        two points can be before the model considers them unrelated
# WhiteKernel -- models observation noise explicitly (real spectroscopic measurements
#        aggregated across many labs/instruments are NOT noise-free, so we should not
#        force the GP to assume perfect measurements)

kernel = ConstantKernel(1.0) * RBF(length_scale=1.0) + WhiteKernel(noise_level=1.0)

gp = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=5,   # re-run hyperparameter optimization from 5 random starts,
                              # since the optimizer can get stuck in local optima otherwise
    random_state=42
)

gp.fit(X_train_scaled, y_train)

print("Fitted kernel:", gp.kernel_)
print("Log-marginal-likelihood:", gp.log_marginal_likelihood(gp.kernel_.theta))

/Users/sahanawaz/miniforge3/envs/chemo_env/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Fitted kernel: 316**2 * RBF(length_scale=1.2e+04) + WhiteKernel(noise_level=1.56e+04)
Log-marginal-likelihood: -34377.578190192784


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

y_pred, y_std = gp.predict(X_test_scaled, return_std=True)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"GP (RBF): RMSE = {rmse:.2f} nm, R² = {r2:.3f}")
print(f"Mean predicted std (uncertainty): {y_std.mean():.2f} nm")
print(f"Std of predicted std (variation in uncertainty across compounds): {y_std.std():.2f} nm")

**Automatic Relevance Determination**

In [ ]:
kernel_ard_quick = (
    ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e3))
    * RBF(length_scale=np.ones(n_features), length_scale_bounds=(1e-2, 1e2))
    + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-3, 1e4))
)

gp_ard_quick = GaussianProcessRegressor(
    kernel=kernel_ard_quick,
    n_restarts_optimizer=2,   # reduced from 5, just to check ARD escapes the degenerate solution
    random_state=42
)

gp_ard_quick.fit(X_train_scaled, y_train)

print("Fitted ARD kernel (quick check):")
print(gp_ard_quick.kernel_)

In [ ]:
import time

start = time.time()
print("Starting GP fit...")

gp_ard_quick.fit(X_train_scaled, y_train)

elapsed = time.time() - start
print(f"Done. Fit took {elapsed:.1f} seconds.")

In [10]:
import time

In [11]:
from sklearn.model_selection import train_test_split

X_train_sub, _, y_train_sub, _ = train_test_split(
    X_train_scaled, y_train, train_size=1500, random_state=42
)

print(f"Subsampled training set: {X_train_sub.shape}")

kernel_ard_quick = (
    ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e3))
    * RBF(length_scale=np.ones(n_features), length_scale_bounds=(1e-2, 1e2))
    + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-3, 1e4))
)

gp_ard_sub = GaussianProcessRegressor(
    kernel=kernel_ard_quick,
    n_restarts_optimizer=1,
    random_state=42
)

start = time.time()
gp_ard_sub.fit(X_train_sub, y_train_sub)
print(f"Done in {time.time() - start:.1f} seconds.")
print(gp_ard_sub.kernel_)

Subsampled training set: (1500, 10)
Done in 30.9 seconds.
31.6**2 * RBF(length_scale=[23.6, 7.26, 100, 100, 100, 20.2, 100, 10.3, 82.6, 14.1]) + WhiteKernel(noise_level=1e+04)


/Users/sahanawaz/miniforge3/envs/chemo_env/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/sahanawaz/miniforge3/envs/chemo_env/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/sahanawaz/miniforge3/envs/chemo_env/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  w

**widen the bounds and rerun the same subsampled check, to see where things actually settle**

In [13]:
kernel_ard_v2 = (
    ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e5))
    * RBF(length_scale=np.ones(n_features), length_scale_bounds=(1e-2, 1e4))
    + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-3, 1e6))
)

gp_ard_v2 = GaussianProcessRegressor(
    kernel=kernel_ard_v2,
    n_restarts_optimizer=1,
    random_state=42
)

start = time.time()
gp_ard_v2.fit(X_train_sub, y_train_sub)
print(f"Done in {time.time() - start:.1f} seconds.")
print(gp_ard_v2.kernel_)

Done in 9.6 seconds.
316**2 * RBF(length_scale=[1e+04, 1e+04, 1e+04, 1e+04, 1e+04, 1e+04, 1e+04, 1e+04, 1e+04, 1e+04]) + WhiteKernel(noise_level=1.65e+04)


/Users/sahanawaz/miniforge3/envs/chemo_env/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/sahanawaz/miniforge3/envs/chemo_env/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/sahanawaz/miniforge3/envs/chemo_env/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better valu

**using the log-marginal-likelihood (LML)**

In [15]:
print(f"Tight-bounds fit LML:  {gp_ard_sub.log_marginal_likelihood(gp_ard_sub.kernel_.theta):.2f}")
print(f"Wide-bounds fit LML:   {gp_ard_v2.log_marginal_likelihood(gp_ard_v2.kernel_.theta):.2f}")

Tight-bounds fit LML:  -9586.85
Wide-bounds fit LML:   -9417.57
